### Setup envs

In [1]:
import boto3
from sagemaker import get_execution_role

In [2]:
!pip install tensorflow==2.5.0

You should consider upgrading via the '/home/ec2-user/anaconda3/envs/python3/bin/python -m pip install --upgrade pip' command.


In [3]:
!pip install tensorflow-recommenders==0.5.2

You should consider upgrading via the '/home/ec2-user/anaconda3/envs/python3/bin/python -m pip install --upgrade pip' command.


In [4]:
role = get_execution_role()
bucket = "ling-cold-start-data"
prefix = "2021-09-22"
data_key = "2021-09-22.csv"
data_location = "s3://{}/{}/{}".format(bucket, prefix, data_key)

In [5]:
import os
import tempfile
from typing import Dict, Text
import pprint 

In [6]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs

### Model definition

In [164]:
class UserModel(tf.keras.Model) :

    def __init__(self, unique_genders, unique_langs, unique_countries, viewer_age, unique_networks):
        super().__init__()

        self.gender_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=unique_genders, mask_token=None),
            tf.keras.layers.Embedding(len(unique_genders) + 1, 4),
        ])

        self.lang_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=unique_langs, mask_token=None),
            tf.keras.layers.Embedding(len(unique_langs) + 1, 10),
        ])

        self.country_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=unique_countries, mask_token=None),
            tf.keras.layers.Embedding(len(unique_countries) + 1, 10),
        ])

        self.network_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=unique_networks, mask_token=None),
            tf.keras.layers.Embedding(len(unique_networks) + 1, 4),
        ])

        age_boundaries = np.array([18, 25, 30, 35, 40, 45, 50, 55, 60, 65, float("inf")])
        self.viewer_age_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.Discretization(age_boundaries.tolist()),
            tf.keras.layers.Embedding(len(age_boundaries), 2)
        ])

        self.viewer_lat_long_embedding = tf.keras.Sequential([
            tf.keras.layers.Lambda(lambda x: tf.numpy_function(self.classify, [x], [tf.float32])),
            tf.keras.layers.Embedding(9, 2)
        ])
    
    @tf.function()
    def call(self, inputs):
        return tf.concat([
            self.gender_embedding(inputs["viewer_gender"]),
            self.lang_embedding(inputs["viewer_lang"]),
            self.country_embedding(inputs["viewer_country"]),
            self.network_embedding(inputs["viewer_network"]),
            self.viewer_age_embedding(inputs["viewer_age"]),
            self.viewer_lat_long_embedding(tf.stack([inputs["viewer_latitude"], inputs["viewer_longitude"]])),
        ], axis = 1)
    
#     @tf.function(input_signature=[tf.TensorSpec([], tf.float32)])
    def classify(self, pair):
        """
        given a datapoint, compute the cluster closest to the datapoint. Return the cluster ID of that cluster.
        :param pair:
        :return: cluster ID
        """
        centroids = np.array(
            [[36.68147669256268, -82.8910274009993],
             [23.22243322909555, 78.23027450833709],
             [50.04997682638993, 0.22379313938744885],
             [37.9309447099281, -117.00741350764692],
             [-32.795864819917725, 148.7159172660312],
             [-18.570548393114084, -54.280255665692565],
             [13.921140442819565, 116.38740315555172],
             [29.78951080730802, 40.279515865947936]]
        )
        res = []
        for datapoint in pair.T:
            dists = np.sqrt(np.sum((centroids - datapoint) ** 2, axis = 1))
            res.append(np.argmin(dists))
        return tf.convert_to_tensor(res)


In [165]:
class BroadcasterModel(tf.keras.Model):

    def __init__(self, unique_movie_titles, dims):
        super().__init__()

        self.broadcaster_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=unique_movie_titles, mask_token=None),
            tf.keras.layers.Embedding(len(unique_movie_titles) + 1, dims)
        ])

    def call(self, broadcaster):
        return tf.concat([
            self.broadcaster_embedding(broadcaster),
        ], axis=1)

### Load data

In [166]:
import tensorflow as tf
import pandas as pd
import numpy as np

def load_data_file_cold(file, stats):
    print('loading file:' + file)
    training_df = pd.read_csv(
        file,
        skiprows=[0],
        names=["viewer","broadcaster","viewer_age","viewer_gender","viewer_longitude","viewer_latitude","viewer_lang","viewer_country","broadcaster_age","broadcaster_gender","broadcaster_longitude","broadcaster_latitude","broadcaster_lang","broadcaster_country","duration", "viewer_network", "broadcaster_network", "count"], dtype={
            'viewer': np.unicode,
            'broadcaster': np.unicode,
            'viewer_age': np.single,
            'viewer_gender': np.unicode,
            'viewer_longitude': np.single,
            'viewer_latitude': np.single,
            'viewer_lang': np.unicode,
            'viewer_country': np.unicode,
            'broadcaster_age': np.single,
            'broadcaster_longitude': np.single,
            'broadcaster_latitude': np.single,
            'broadcaster_lang': np.unicode,
            'broadcaster_country': np.unicode,
            'viewer_network': np.unicode,
            'broadcaster_network': np.unicode,
            'count': np.unicode,
        })
    
    values = {
        'viewer': 'unknown',
        'broadcaster': 'unknown',
        'viewer_age': 30,
        'viewer_gender': 'unknown',
        'viewer_longitude': 0,
        'viewer_latitude': 0,
        'viewer_lang': 'unknown',
        'viewer_country': 'unknown',
        'broadcaster_age': 30,
        'broadcaster_longitude': 0,
        'broadcaster_latitude': 0,
        'broadcaster_lang': 'unknown',
        'broadcaster_country': 'unknown',
        'duration': 0,
        'viewer_network': 'unknown',
        'broadcaster_network': 'unknown',
        'count': '0'
    }

    training_df.fillna(value=values, inplace=True)
    print(training_df.head(10))
    print(training_df.iloc[-10:])
#     stats.send_stats('data-size', len(training_df.index))
    training_df = training_df.sample(frac=.001)
    return training_df


def load_training_data_cold(file, stats):
    ratings_df = load_data_file_cold(file, stats)
    print('creating data set')
    training_ds = (
        tf.data.Dataset.from_tensor_slices(
            ({
                "viewer": tf.cast(
                    ratings_df['viewer'].values,
                    tf.string),
                "viewer_gender": tf.cast(
                    ratings_df['viewer_gender'].values,
                    tf.string),
                "viewer_lang": tf.cast(
                    ratings_df['viewer_lang'].values,
                    tf.string),
                "viewer_country": tf.cast(
                    ratings_df['viewer_country'].values,
                    tf.string),
                "viewer_age": tf.cast(
                    ratings_df['viewer_age'].values,
                    tf.int16),
                "viewer_longitude": tf.cast(
                    ratings_df['viewer_longitude'].values,
                    tf.float16),
                "viewer_latitude": tf.cast(
                    ratings_df['viewer_latitude'].values,
                    tf.float16),
                "broadcaster": tf.cast(
                    ratings_df['broadcaster'].values,
                    tf.string),
                "viewer_network": tf.cast(
                    ratings_df['viewer_network'].values,
                    tf.string),
                "broadcaster_network": tf.cast(
                    ratings_df['broadcaster_network'].values,
                    tf.string),
            })))

    return training_ds


def prepare_training_data_cold(train_ds):
    print('prepare_training_data')
    training_ds = train_ds.cache().map(lambda x: {
        "broadcaster": x["broadcaster"],
        "viewer": x["viewer"],
        "viewer_gender": x["viewer_gender"],
        "viewer_lang": x["viewer_lang"],
        "viewer_country": x["viewer_country"],
        "viewer_age": x["viewer_age"],
        "viewer_longitude": x["viewer_longitude"],
        "viewer_latitude": x["viewer_latitude"],
        "viewer_network": x["viewer_network"],
        "broadcaster_network": x["broadcaster_network"],
    }, num_parallel_calls=tf.data.AUTOTUNE,
       deterministic=False)

    print('done prepare_training_data')
    return training_ds


In [167]:
def get_broadcaster_data_set(train_ds):
    broadcasters = train_ds.cache().map(lambda x: x["broadcaster"], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    broadcasters_ds = tf.data.Dataset.from_tensor_slices(
        np.unique(list(broadcasters.as_numpy_iterator())))
    return broadcasters_ds

In [168]:
training_dataset = load_training_data_cold(file=data_location, stats="")

loading file:s3://ling-cold-start-data/2021-09-22/2021-09-22.csv
                                            viewer  \
0  cd 43 8e ef e8 29 d8 a4 b0 4b ad aa d2 b2 0d 4b   
1  70 b9 f2 5a 7d 06 13 4b 98 8a ff ee f7 6f ee 20   
2  7a 17 b5 e4 af 50 29 0b 00 93 38 fd 66 fc 33 c4   
3  a3 d1 a2 3e 11 28 e1 90 98 6c 62 80 ee 95 bd fd   
4  9f ce 40 d8 ee 57 f3 d8 d7 f0 b0 45 25 24 d4 c7   
5  90 a9 c6 3f 28 43 df 13 2b 6b 05 2e 9e 1d 94 dd   
6  44 87 40 b4 65 5b 58 bb 46 b1 49 2f 20 9e 51 67   
7  9c d4 2b 45 85 f9 24 14 eb 3e c9 f8 3c 22 f3 36   
8  0c 8f 39 1b ec be ce 07 71 a1 14 19 e4 a0 38 1c   
9  5b a0 43 af 9c 29 5a 77 67 52 fd f2 c9 f4 6c 73   

                                       broadcaster  viewer_age viewer_gender  \
0  da 46 7f 4e ba 5a da ac 99 85 99 04 97 be 27 f3        21.0        female   
1  28 a2 1e ba 79 20 9d 20 72 1e 63 f6 6d d9 9e b7        24.0        female   
2  86 99 d1 be 2a a1 06 0d e6 75 69 52 62 14 4f c8        19.0          male   
3  e4 20 83 74 75 f9

In [169]:
train = prepare_training_data_cold(training_dataset)

prepare_training_data
done prepare_training_data


/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/data/ops/dataset_ops.py:3704: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable.debug_mode()`.
  "Even though the `tf.config.experimental_run_functions_eagerly` "


In [170]:
broadcasters_data_set = get_broadcaster_data_set(training_dataset)

### Prepare features

In [171]:
def get_list(training_data, key):
    return training_data.batch(1_000_000).map(lambda x: x[key], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)


def get_unique_list(data):
    return np.unique(np.concatenate(list(data)))

In [172]:
user_genders = get_list(train, 'viewer_gender')

In [173]:
user_langs = get_list(train, 'viewer_lang')

In [174]:
user_countries = get_list(train, 'viewer_country')

In [175]:
viewer_age = get_list(train, 'viewer_age')

In [176]:
user_networks = get_list(train, 'viewer_network')

### derive input dims

In [177]:
unique_user_genders = get_unique_list(user_genders)

In [178]:
len(unique_user_genders)

2

In [179]:
unique_user_langs = get_unique_list(user_langs)

In [180]:
len(unique_user_langs)

25

In [181]:
unique_user_countries = get_unique_list(user_countries)

In [182]:
len(unique_user_countries)

54

In [183]:
unique_user_networks = get_unique_list(user_networks)

In [184]:
len(unique_user_networks)

4

### user model

In [185]:
age_boundaries = np.array([18, 25, 30, 35, 40, 45, 50, 55, 60, 65, float("inf")])

In [186]:
user_model = UserModel(unique_user_genders, unique_user_langs, unique_user_countries, viewer_age, unique_user_networks)

### broadcaster model

In [187]:
broadcaster_ids = get_list(train, 'broadcaster')

In [188]:
unique_broadcasters = get_unique_list(broadcaster_ids)

In [189]:
len(unique_broadcasters)

4417

In [190]:
broadcaster_embedding_dimension = 32

In [191]:
broadcaster_model = BroadcasterModel(unique_broadcasters, broadcaster_embedding_dimension)

### two tower model

In [192]:
metrics = tfrs.metrics.FactorizedTopK(candidates=broadcasters_data_set.batch(128).map(broadcaster_model))

In [193]:
task = tfrs.tasks.Retrieval(
    metrics=metrics
)

In [194]:
class TwoTowers(tf.keras.Model):

    def __init__(self, broadcaster_model, user_model, task):
        super().__init__()
        self.broadcaster_model: tf.keras.Model = broadcaster_model
        self.embedding_model = user_model
        self.task: tf.keras.layers.Layer = task

    def train_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:

        # Set up a gradient tape to record gradients.
        with tf.GradientTape() as tape:

            # Loss computation.

            user_embeddings = self.embedding_model({
                "viewer_gender": features["viewer_gender"],
                "viewer_lang": features["viewer_lang"],
                "viewer_country": features["viewer_country"],
                "viewer_age": features["viewer_age"],
                "viewer_network": features["viewer_network"],
                "viewer_latitude": features["viewer_latitude"],
                "viewer_longitude": features["viewer_longitude"]
            })
            positive_broadcaster_embeddings = self.broadcaster_model(
                features["broadcaster"])
            loss = self.task(user_embeddings, positive_broadcaster_embeddings)

            # Handle regularization losses as well.
            regularization_loss = sum(self.losses)

            total_loss = loss + regularization_loss

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_variables))

        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss

        return metrics

    def test_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:

        # Loss computation.

        user_embeddings = self.embedding_model({
            "viewer": features["viewer"],
        })
        positive_broadcaster_embeddings = self.broadcaster_model(
            features["broadcaster"])
        loss = self.task(user_embeddings, positive_broadcaster_embeddings)

        # Handle regularization losses as well.
        regularization_loss = sum(self.losses)

        total_loss = loss + regularization_loss

        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        return metrics

In [195]:
model = TwoTowers(broadcaster_model, user_model, task)

In [196]:
learning_rate = 0.05
batch_size = 16384
# batch_size = 250
epochs = 2
top_k = 1999

In [197]:
tf.config.run_functions_eagerly(True)

In [198]:
model.compile(
    optimizer=tf.keras.optimizers.Adagrad(learning_rate=learning_rate),
    run_eagerly=True)

In [199]:
train_ds = train.batch(batch_size).cache()
# train_ds = train_ds.prefetch(tf.data.experimental.AUTOTUNE)

In [200]:
model.fit(train_ds, epochs=1)

1/1 [==============================] - 2s 2s/step - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 3.8506e-04 - factorized_top_k/top_10_categorical_accuracy: 0.0021 - factorized_top_k/top_50_categorical_accuracy: 0.0116 - factorized_top_k/top_100_categorical_accuracy: 0.0210 - loss: 44435.9453 - regularization_loss: 0.0000e+00 - total_loss: 44435.9453


In [201]:
data_location

's3://ling-cold-start-data/2021-09-22/2021-09-22.csv'

In [202]:
from datetime import date

In [203]:
model_location = "s3://{}/{}/{}".format(bucket, prefix, date.today())

In [204]:
model_location

's3://ling-cold-start-data/2021-09-22/2021-10-14'

In [205]:
tf.config.run_functions_eagerly(True)
print("create index")
index = tfrs.layers.factorized_top_k.BruteForce(
    query_model=user_model,
    k=top_k,
)

index.index(
    broadcasters_data_set.batch(10000).map(
        model.broadcaster_model),
    broadcasters_data_set)

_, titles = index(
    {
        "viewer_gender": tf.constant(["male"]),
        "viewer_lang": tf.constant(["en"]),
        "viewer_country": tf.constant(["US"]),
        "viewer_age": tf.constant([38]),
        "viewer_longitude": tf.constant([-74.89611]),
        "viewer_latitude": tf.constant([40.36393]),
        "viewer_network": tf.constant(["meetme"]),
        "viewer_lat_long_cluster": tf.constant(["7"]),
    }
)

print(f"Recommendations for user lam: {titles}")

_, titles = index(
    {
        "viewer_gender": tf.constant(["male"]),
        "viewer_lang": tf.constant(["en"]),
        "viewer_country": tf.constant(["US"]),
        "viewer_age": tf.constant([28]),
        "viewer_longitude": tf.constant([-118.41625]),
        "viewer_latitude": tf.constant([34.10313]),
        "viewer_network": tf.constant(["pof"]),
        "viewer_lat_long_cluster": tf.constant(["5"]),
    }
)

print(f"Recommendations for user cal: {titles}")

_, titles = index(
    {
        "viewer_gender": tf.constant(["female"]),
        "viewer_lang": tf.constant(["en"]),
        "viewer_country": tf.constant(["US"]),
        "viewer_age": tf.constant([32]),
        "viewer_longitude": tf.constant([-74.89611]),
        "viewer_latitude": tf.constant([40.36393]),
        "viewer_network": tf.constant(["skout"]),
        "viewer_lat_long_cluster": tf.constant(["7"]),
    }
)

print(f"Recommendations for user 32: {titles}")

create index


Recommendations for user lam: [[b'bc 8b 7a f9 34 0e b9 bf 7a b0 d4 89 70 79 ea c0'
  b'54 e5 7d c3 25 cb 79 5d ad 09 04 07 66 9d f6 00'
  b'5f 5e bf f9 c9 5a d2 00 0d e6 92 3f ad 23 11 88' ...
  b'f1 29 a1 81 93 b7 2d c9 d7 5e a1 68 37 e7 19 f5'
  b'e5 f2 a4 df 41 62 c6 2a ff e3 5b e3 a3 24 56 21'
  b'22 3d 28 42 0d 98 3d 52 a5 d5 d2 c0 30 8f d3 8d']]


Recommendations for user cal: [[b'0d fc 9e 53 5a 0d dd 98 d9 5c 9e de 23 f6 ed f9'
  b'5e ae 09 39 7f 25 cc 52 f5 06 53 3f 09 23 c0 c6'
  b'5b 02 80 ba a6 22 98 55 0d 02 78 03 c6 47 a0 88' ...
  b'ae d5 58 2a 45 54 85 20 b0 e5 7b 1a e1 e1 9e c7'
  b'81 eb 02 69 86 51 96 30 a2 2a 41 c8 84 c8 1a 46'
  b'b0 7a 08 fe ae 56 e8 ca fd d7 f8 2a af 6c 35 2b']]


Recommendations for user 32: [[b'52 14 17 f4 43 24 0a f0 1d 3f 44 97 a4 37 47 69'
  b'e0 ab 9e 0d 7c df 32 95 58 fe 55 05 5a 44 fb 7d'
  b'6a 71 a5 f1 80 9f cd 44 6a b6 25 56 fc c3 79 5f' ...
  b'ab 5f 4f f5 43 94 0a 62 46 0b 14 f2 b3 fa f5 47'
  b'e7 74 f3 de 03 c9 d0 39 69 01 84 50 90 10 4c 93'
  b'3f 80 eb 13 c7 fb 71 7a 41 b9 40 80 ad 38 be 00']]


In [206]:
tf.saved_model.save(
      index,
      model_location,
      options=tf.saved_model.SaveOptions(namespace_whitelist=None)
  )

AttributeError: in user code:

    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/keras/saving/saving_utils.py:130 _wrapped_model  *
        outputs = model(inputs, training=False)
    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow_recommenders/layers/factorized_top_k.py:547 call  *
        queries = self.query_model(queries)
    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/keras/engine/base_layer.py:1030 __call__  **
        outputs = call_fn(inputs, *args, **kwargs)
    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/eager/def_function.py:866 __call__
        return self._python_function(*args, **kwds)
    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/eager/function.py:3966 bound_method_wrapper
        return wrapped_fn(weak_instance(), *args, **kwargs)
    <ipython-input-164-2695a39d0cd4>:49 call
        s

In [133]:
index.save(model_location)
# index.save(model_location, save_format='tf')
# tf.saved_model.save(index, model_location)
# tf.keras.models.save_model(index, model_location)

AttributeError: in user code:

    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/keras/saving/saving_utils.py:130 _wrapped_model  *
        outputs = model(inputs, training=False)
    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow_recommenders/layers/factorized_top_k.py:547 call  *
        queries = self.query_model(queries)
    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/keras/engine/base_layer.py:1030 __call__  **
        outputs = call_fn(inputs, *args, **kwargs)
    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/eager/def_function.py:866 __call__
        return self._python_function(*args, **kwds)
    /home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/tensorflow/python/eager/function.py:3966 bound_method_wrapper
        return wrapped_fn(weak_instance(), *args, **kwargs)
    <ipython-input-116-fe09901ccf88>:49 call
        s

In [92]:
tf.saved_model.DEFAULT_SERVING_SIGNATURE_DEF_KEY

'serving_default'

In [95]:
class CustomModule(tf.Module):

    def __init__(self):
        super(CustomModule, self).__init__()
        self.v = tf.Variable(1.)

    @tf.function
    def __call__(self, x):
        print('Tracing with', x)
        return x * self.v

    @tf.function(input_signature=[tf.TensorSpec([], tf.float32)])
    def mutate(self, new_v):
        self.v.assign(new_v)

module = CustomModule()

In [96]:
module_no_signatures_path = os.path.join('module_no_signatures')
module(tf.constant(0.))
print('Saving model...')
tf.saved_model.save(module, module_no_signatures_path)

Tracing with tf.Tensor(0.0, shape=(), dtype=float32)
Saving model...
INFO:tensorflow:Assets written to: module_no_signatures/assets


INFO:tensorflow:Assets written to: module_no_signatures/assets
